In [ ]:
# ============================
# Phase A — DATASET GENERATION
# ============================

# Cell 1 — Imports & setup
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Dict, Literal

# Reproducibility
RNG = np.random.default_rng(42)

DATA_DIR = Path("./tm_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Cell 2 — Topologies + TM generators

@dataclass
class TopologySpec:
    name: str
    G: nx.Graph
    capacities: Dict[tuple, float]  # key = sorted(u,v)
    seed: int = None

def square_topology(capacity: float = 10.0) -> TopologySpec:
    G = nx.Graph()
    G.add_edges_from([(1,2),(2,3),(3,4),(4,1)])
    caps = {tuple(sorted(e)): capacity for e in G.edges()}
    return TopologySpec("square4", G, caps)

def cycle_topology(n: int = 8, capacity: float = 10.0) -> TopologySpec:
    """General cycle of n nodes (n >= 3)."""
    G = nx.cycle_graph(n)
    G = nx.relabel_nodes(G, {i: i+1 for i in range(n)})  # label nodes 1..n
    caps = {tuple(sorted(e)): capacity for e in G.edges()}
    return TopologySpec(f"cycle{n}", G, caps)


def abilene_like(capacity_range: Tuple[float,float] = (8.0, 12.0)) -> TopologySpec:
    G = nx.Graph()
    G.add_nodes_from(range(1, 13))
    edges = [
        (1,2),(2,3),(3,4),(4,5),(5,6),
        (6,7),(7,8),(8,9),(9,10),(10,11),(11,12),(12,1),
        (2,6),(3,7),(5,9),(8,12),(4,10)
    ]
    G.add_edges_from(edges)
    low, high = capacity_range
    caps = {tuple(sorted(e)): float(RNG.uniform(low, high)) for e in G.edges()}
    return TopologySpec("abilene12_synth", G, caps)

def rocketfuel_like(n_nodes: int = 30, p_backbone: float = 0.1,
                    p_access: float = 0.05,
                    capacity_range: Tuple[float,float] = (6.0, 14.0)) -> TopologySpec:
    nb = max(8, n_nodes // 3)
    # connected backbone
    while True:
        seed = int(RNG.integers(1<<30))   # one seed per attempt
        backbone = nx.erdos_renyi_graph(nb, p_backbone, seed=seed)
        if nx.is_connected(backbone):
            print(f"[Rocketfuel-like] Using seed {seed} for backbone")
            break
    mapping = {i: i+1 for i in range(nb)}
    backbone = nx.relabel_nodes(backbone, mapping)
    G = nx.Graph(); G.update(backbone)
    # attach/access
    for v in range(nb+1, n_nodes+1):
        deg_targets = RNG.choice(list(range(1, nb+1)),
                                 size=int(RNG.integers(1, 4)),
                                 replace=False)
        for t in deg_targets:
            G.add_edge(v, int(t))
    # sparse access-access
    access_nodes = list(range(nb+1, n_nodes+1))
    for _ in range(int(0.03 * len(access_nodes) * (len(access_nodes)-1) / 2)):
        u, v = RNG.choice(access_nodes, size=2, replace=False)
        if not G.has_edge(int(u), int(v)) and RNG.random() < p_access:
            G.add_edge(int(u), int(v))
    low, high = capacity_range
    caps = {tuple(sorted(e)): float(RNG.uniform(low, high)) for e in G.edges()}
    return TopologySpec(f"rf_{n_nodes}_synth", G, caps, seed=seed)


# TM generators
def zero_diag(M: np.ndarray) -> np.ndarray:
    M = M.copy(); np.fill_diagonal(M, 0.0); return M

def gen_uniform(N: int, low: float = 0.5, high: float = 5.0, rng=RNG) -> np.ndarray:
    return zero_diag(rng.uniform(low, high, size=(N, N)))

def gen_exponential(N: int, scale: float = 4.0, rng=RNG) -> np.ndarray:
    return zero_diag(rng.exponential(scale, size=(N, N)))

def gen_gravity(N: int, mu: float = 0.0, sigma: float = 1.0,
                noise_low: float = 0.8, noise_high: float = 1.2, rng=RNG) -> np.ndarray:
    w = rng.lognormal(mean=mu, sigma=sigma, size=N)
    TM = np.outer(w, w) * rng.uniform(noise_low, noise_high, size=(N, N))
    return zero_diag(TM)

def total_capacity(G: nx.Graph, capacities: Dict[tuple, float]) -> float:
    return float(sum(capacities[tuple(sorted(e))] for e in G.edges()))

def scale_tm(TM: np.ndarray, G: nx.Graph, capacities: Dict[tuple, float],
             load_level: float = 0.7, kappa: float = 3.0) -> np.ndarray:
    C_tot = total_capacity(G, capacities)
    D_target = kappa * load_level * C_tot
    curr_sum = float(TM.sum())
    if curr_sum <= 1e-9: return TM
    return TM * (D_target / curr_sum)

from typing import Literal
GenName = Literal["uniform", "exponential", "gravity"]

def sample_tm(N: int, topo: TopologySpec,
              mix: Dict[GenName, float] = {"exponential":0.6,"gravity":0.3,"uniform":0.1},
              load_levels: Tuple[float,...] = (0.3,0.5,0.7,0.9,1.1),
              kappa_range: Tuple[float,float] = (2.0, 4.0)) -> tuple:
    gens = list(mix.keys())
    probs = np.array([mix[g] for g in gens], dtype=float); probs = probs / probs.sum()
    gname = str(RNG.choice(gens, p=probs))
    TM = gen_uniform(N) if gname=="uniform" else gen_exponential(N) if gname=="exponential" else gen_gravity(N)
    L = float(RNG.choice(load_levels))
    kappa = float(RNG.uniform(*kappa_range))
    TM_scaled = scale_tm(TM, topo.G, topo.capacities, L, kappa)
    return TM_scaled, gname, L, kappa


In [ ]:
# Cell 3 — Choose topology & dataset config

# TOPO = square_topology(10.0)   # old 4-node square
TOPO = rocketfuel_like(n_nodes=30)

N = TOPO.G.number_of_nodes()

N_SAMPLES = 2000
mix_cfg = {"exponential":0.6, "gravity":0.3, "uniform":0.1}
load_levels = (0.3, 0.5, 0.7, 0.9, 1.1)
kappa_range = (2.0, 4.0)


In [ ]:
# Cell 4 — Build & save dataset + topology

all_TMs = np.zeros((N_SAMPLES, N, N), dtype=float)
meta_rows = []
for i in range(N_SAMPLES):
    TM, gname, L, kappa = sample_tm(N, TOPO, mix=mix_cfg, load_levels=load_levels, kappa_range=kappa_range)
    all_TMs[i] = TM
    meta_rows.append((gname, L, kappa))

# Resolve final topology name (append seed if Rocketfuel)
# topo_name = TOPO.name
# if "rf_" in topo_name:
#     try:
#         topo_name = f"{TOPO.name}_seed{seed}"   # seed must be captured in rocketfuel_like()
#     except NameError:
#         print("[Warning] No seed variable found — filenames will not include seed.")
        
topo_name = TOPO.name
if TOPO.seed is not None:   # only rocketfuel-like has a seed
    topo_name = f"{TOPO.name}_seed{TOPO.seed}"


# Save TMs and metadata
tm_npy = DATA_DIR / f"{topo_name}_TMs.npy"
tm_csv = DATA_DIR / f"{topo_name}_TMs_meta.csv"
np.save(tm_npy, all_TMs)
pd.DataFrame(meta_rows, columns=["gen","load_level","kappa"]).to_csv(tm_csv, index=False)

# Save topology
edges_sorted = [tuple(sorted(e)) for e in TOPO.G.edges()]
caps_ordered = [TOPO.capacities[e] for e in edges_sorted]
topo_npz = DATA_DIR / f"{topo_name}_topology.npz"
np.savez(topo_npz,
         name=TOPO.name,  # keep original base name here
         nodes=np.array(list(TOPO.G.nodes()), dtype=int),
         edges=np.array(edges_sorted, dtype=int),
         capacities=np.array(caps_ordered, dtype=float))


In [ ]:
# Cell 5 — Quick preview + save one TM CSV

TMS = np.load(tm_npy)
META = pd.read_csv(tm_csv)
print("Dataset shape:", TMS.shape)
print(META.head())

rows = []
for idx in np.random.choice(len(TMS), size=3, replace=False):
    rows.append({
        "idx": int(idx),
        "gen": META.loc[idx, "gen"],
        "load_level": float(META.loc[idx, "load_level"]),
        "kappa": float(META.loc[idx, "kappa"]),
        "sum_demand": float(TMS[idx].sum()),
        "max_demand": float(TMS[idx].max())
    })
print(pd.DataFrame(rows))

example_path = DATA_DIR / f"{TOPO.name}_example_TM.csv"
pd.DataFrame(TMS[int(rows[0]['idx'])]).to_csv(example_path, index=False)
print(f"[Saved Example TM] {example_path}")


In [ ]:
npz = np.load(topo_npz, allow_pickle=True)
nodes = npz["nodes"].astype(int).tolist()
edges = [tuple(map(int, e)) for e in npz["edges"]]
capacities_vals = npz["capacities"].astype(float).tolist()

print(f"[Reloaded Topology] {npz['name']}")
print("Nodes:", len(nodes), "Edges:", len(edges))
print("Sample capacities:", capacities_vals[:5])


In [ ]:
# =======================================
# Phase B — RL TRAINING & TRANSFER TESTS
# =======================================

# Cell 0 — Load dataset, split, rebuild graph state, and precompute/load paths

import numpy as np, pandas as pd, networkx as nx, pickle
from pathlib import Path
from networkx.algorithms.simple_paths import shortest_simple_paths

DATA_DIR = Path("./tm_dataset")
TOPO_NAME = "rf_30_synth_seed438654291"

# --- Load traffic matrices + metadata ---
TMS = np.load(DATA_DIR / f"{TOPO_NAME}_TMs.npy")
META = pd.read_csv(DATA_DIR / f"{TOPO_NAME}_TMs_meta.csv")

# --- Load topology ---
topo_npz = np.load(DATA_DIR / f"{TOPO_NAME}_topology.npz", allow_pickle=True)
nodes = topo_npz["nodes"].astype(int).tolist()
edges_arr = topo_npz["edges"]
edges = [tuple(map(int, e)) for e in edges_arr]
capacities_vals = topo_npz["capacities"].tolist()

# Build graph + capacities
G = nx.Graph(); G.add_nodes_from(nodes); G.add_edges_from(edges)
cap_dict = {tuple(sorted(e)): float(c) for e, c in zip(edges, capacities_vals)}
EDGE_LIST = [tuple(sorted(e)) for e in G.edges()]
EDGE_INDEX = {e:i for i,e in enumerate(EDGE_LIST)}

# --- Train/test split ---
n_total = len(TMS); n_train = int(0.7 * n_total)
TMS_train, META_train = TMS[:n_train], META.iloc[:n_train].reset_index(drop=True)
TMS_test,  META_test  = TMS[n_train:],  META.iloc[n_train:].reset_index(drop=True)

# --- Candidate paths (with caching + parallel + progress) ---
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import pickle

k_paths = 3   # top-k shortest simple paths
paths_file = DATA_DIR / f"{TOPO_NAME}_candidate_paths.pkl"

def compute_paths(args):
    src, dst, G, k_paths = args
    paths = list(shortest_simple_paths(G, src, dst))[:k_paths]
    return (src, dst), paths

if paths_file.exists():
    print(f"[Info] Loading candidate paths from {paths_file}")
    with open(paths_file, "rb") as f:
        candidate_paths = pickle.load(f)
else:
    print("[Info] Precomputing candidate paths in parallel...")
    pairs = [(s, d, G, k_paths) for s in nodes for d in nodes if s != d]

    candidate_paths = {}
    with ProcessPoolExecutor() as ex:
        for (src, dst), paths in tqdm(
            ex.map(compute_paths, pairs), total=len(pairs), desc="Src-Dst pairs"
        ):
            candidate_paths[(src, dst)] = paths

    # Save for reuse
    with open(paths_file, "wb") as f:
        pickle.dump(candidate_paths, f)
    print(f"[Info] Saved candidate paths to {paths_file}")

print(f"Candidate paths ready for {len(candidate_paths)} src-dst pairs "
      f"(up to {k_paths} per pair).")

import random

# --- Verification: sample a few random pairs ---
sample_pairs = random.sample(list(candidate_paths.keys()), 5)
print("\n[Verification] Sample candidate paths:")
for (s, d) in sample_pairs:
    paths = candidate_paths[(s, d)]
    print(f"  Pair ({s} → {d}):")
    for i, p in enumerate(paths, 1):
        print(f"    Path {i}: {p}")

# --- Coverage check ---
counts = [len(v) for v in candidate_paths.values()]
print("\n[Sanity] Path count distribution:")
print(f"  Min paths: {min(counts)}")
print(f"  Max paths: {max(counts)}")
print(f"  Pairs with <{k_paths} paths: {sum(c < k_paths for c in counts)} / {len(counts)}")


In [ ]:
# Cell 1 — Torch config & dimensions

import torch, torch.nn as nn, torch.optim as optim, random
from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

N = TMS.shape[1]
N_ACTIONS = N * (N - 1)                     # all ordered src->dst (dst!=src)
STATE_DIM = N*N + N_ACTIONS + len(EDGE_LIST)  # TM + mask + link loads

print(f"N={N}, Actions={N_ACTIONS}, Edges={len(EDGE_LIST)}, STATE_DIM={STATE_DIM}")


In [ ]:
# Cell 2 — φ(s) encoder + DS-DQN (two-time-scale via learning rates)

# Cell 2 — φ(s) encoder + DS-DQN (two-time-scale via learning rates)

class PhiEncoder(nn.Module):
    def __init__(self, state_dim, phi_dim=None):
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim   # default: no compression

        # You can still keep hidden layers, but final size = phi_dim (= state_dim)
        self.fc1 = nn.Linear(state_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, phi_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)  # φ(s)

# Debug: check encoder output dimension
phi_encoder = PhiEncoder(STATE_DIM, phi_dim=None).to(device)
dummy = torch.zeros(1, STATE_DIM).to(device)
out = phi_encoder(dummy)
print("Input dim:", dummy.shape, "Output dim:", out.shape)

class DS_DQN(nn.Module):
    def __init__(self,
                 state_dim,
                 n_actions,
                 phi_dim=None,      # default None → will become = state_dim
                 lr_enc=5e-5,       # α (slow)
                 lr_w=3e-3):        # β (fast)
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim   # keep φ(s) same dim as raw state

        self.encoder = PhiEncoder(state_dim, phi_dim)

        # linear head: action weights match phi_dim
        self.w = nn.Parameter(torch.randn(n_actions, phi_dim) * 0.01)

        # separate optimizers
        self.opt_enc = optim.Adam(self.encoder.parameters(), lr=lr_enc)
        self.opt_w   = optim.Adam([self.w], lr=lr_w)

    def Q(self, s):
        """Return Q-values and features: Q(s,·) = φ(s) @ w^T."""
        phi = self.encoder(s)                 # [B, d]
        q = torch.matmul(phi, self.w.T)       # [B, A]
        return q, phi


    @staticmethod
    def _pearson_corr_pairwise(phi, i, j, eps=1e-8):
        """Pearson correlation between two embeddings φ_i, φ_j."""
        x = phi[i]; y = phi[j]
        xm = x - x.mean(); ym = y - y.mean()
        num = (xm * ym).sum()
        den = torch.sqrt((xm * xm).sum() + eps) * torch.sqrt((ym * ym).sum() + eps)
        return num / den

    def _corr_penalty_state_pairs(self, phi, m_pairs=16):
        """L2(θ1): average of squared positive Pearson correlations across m random pairs."""
        B, _ = phi.shape
        if B < 2:
            return torch.tensor(0.0, device=phi.device)

        max_pairs = B * (B - 1) // 2
        m = int(min(m_pairs, max_pairs))
        if m <= 0:
            return torch.tensor(0.0, device=phi.device)

        idx_i = torch.randint(0, B, (m,), device=phi.device)
        idx_j = torch.randint(0, B, (m,), device=phi.device)
        mask = (idx_i != idx_j)
        if mask.sum() == 0:
            return torch.tensor(0.0, device=phi.device)
        idx_i, idx_j = idx_i[mask], idx_j[mask]

        r_list = []
        for ii, jj in zip(idx_i, idx_j):
            r = self._pearson_corr_pairwise(phi, ii, jj)
            r_list.append(r)  # penalize positive correlation
        if len(r_list) == 0:
            return torch.tensor(0.0, device=phi.device)
        r_stack = torch.stack(r_list)
        return (r_stack ** 2).mean()

    def update(self, batch, target_net, gamma=0.99, lambda_reg=0.0):
        """
        One DS-DQN update:
          - L1: TD loss on Q(s,a) (affects both w and θ1)
          - L2: correlation penalty on φ(s) (affects θ1 only)
          - Two-time-scale: w fast (lr_w), θ1 slow (lr_enc), both updated every step
        """
        s, a, r, s_next, done = batch
        s      = torch.FloatTensor(s).to(device)
        s_next = torch.FloatTensor(s_next).to(device)
        a      = torch.LongTensor(a).to(device)
        r      = torch.FloatTensor(r).to(device)
        done   = torch.FloatTensor(done).to(device)

        # Current Q and picked action value
        q_vals, phi = self.Q(s)
        q_sa = q_vals.gather(1, a.unsqueeze(1)).squeeze(1)

        # Bellman target via target network
        with torch.no_grad():
            q_next, _ = target_net.Q(s_next)
            y = r + gamma * (1 - done) * q_next.max(1)[0]

        # L1: TD error
        td_loss = (y - q_sa).pow(2).mean()

        # L2: decorrelation loss
        l2_loss = self._corr_penalty_state_pairs(phi, m_pairs=16)

        # Total loss
        loss = td_loss + lambda_reg * l2_loss

        # Backprop once
        self.opt_enc.zero_grad()
        self.opt_w.zero_grad()
        loss.backward()

        # Step BOTH every update (two-time-scale via lr sizes)
        self.opt_w.step()    # fast
        self.opt_enc.step()  # slow

        return td_loss.item(), l2_loss.item()


In [ ]:
# Cell 3 — Replay buffer

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    def sample(self, batch_size=32):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = map(np.array, zip(*batch))
        return s, a, r, s_next, done
    def __len__(self):
        return len(self.buffer)


In [ ]:
# Cell 4 — Environment helpers (state, action mapping, reward, with τ filtering)

def flow_index_to_pair(a: int, N: int) -> tuple:
    src = a // (N - 1)
    dst = a % (N - 1)
    if dst >= src: 
        dst += 1
    return src, dst  # 0-based (align with TM indexing)


def init_state(TM):
    mask = np.zeros(N_ACTIONS, dtype=float)
    loads = np.zeros(len(EDGE_LIST), dtype=float)
    return np.concatenate([TM.flatten(), mask, loads])


def apply_action(state, action, TM, lam=0.5, tau=0.8):
    """
    One environment step:
    - Pick action (src,dst)
    - Route demand using precomputed candidate paths[(src,dst)]
    - Apply τ-threshold filtering on link utilizations
    - Update loads and compute reward
    """
    TM_flat = state[:N*N]
    mask = state[N*N:N*N+N_ACTIONS]
    loads = state[N*N+N_ACTIONS:]

    TM_mat = TM_flat.reshape(N, N).copy()
    if mask[action] == 1.0:
        # Already routed this flow → small penalty
        U = max(loads[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST)))
        rho = np.mean([loads[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST))])
  #      r = (1-lam)*(1-U) - lam*rho - 0.1
        r = (1 - lam) * (1 - U) + lam * (1 - rho) - 0.1
        return state.copy(), r, U, rho

    # Mark this flow as routed
    mask_new = mask.copy()
    mask_new[action] = 1.0

    s, d = flow_index_to_pair(action, N)
    demand = TM_mat[s, d]

    # --- Path selection with τ filtering ---
    paths = candidate_paths[(s+1, d+1)]  # precomputed (1-based nodes in G)
    chosen_path = None

    for path in paths:  # check each candidate path
        safe = True
        for u, v in zip(path[:-1], path[1:]):
            e = tuple(sorted((u, v))); idx = EDGE_INDEX[e]
            projected_util = (loads[idx] + demand) / cap_dict[e]
            if projected_util > tau:
                safe = False
                break
        if safe:
            chosen_path = path
            break

    # If no safe path, fallback: pick first candidate anyway (with penalty)
    if chosen_path is None:
        chosen_path = paths[0]

    # --- Update link loads ---
    loads_new = loads.copy()
    for u, v in zip(chosen_path[:-1], chosen_path[1:]):
        e = tuple(sorted((u, v))); idx = EDGE_INDEX[e]
        loads_new[idx] += demand

    # --- Compute utilization & reward ---
    utilizations = np.array([loads_new[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST))])
    U = float(utilizations.max())
    rho = float(utilizations.mean())
  #  r = (1 - lam) * (1 - U) - lam * rho
    r = (1 - lam) * (1 - U) + lam * (1 - rho)


    s_next = np.concatenate([TM_mat.flatten(), mask_new, loads_new])
    return s_next, r, U, rho


In [ ]:
# Cell 4a — Quick debug check
state0 = init_state(TMS_train[0])
s_next, r, U, rho = apply_action(state0, action=0, TM=TMS_train[0], lam=0.55, tau=0.8)

print("STATE_DIM check:", STATE_DIM)
print("Initial state shape:", state0.shape)
print("Next state shape:", s_next.shape)
print("Reward example:", r, "U:", U, "rho:", rho)


In [ ]:
# Cell 5 — Training loop (DS-DQN with ε-decay + τ filtering paths)

EPISODES = 3000          # longer training
K = 30                    # steps per episode
BATCH_SIZE = 32
GAMMA = 0.99
TARGET_UPDATE = 50

# ε-greedy schedule
eps_start = 1.0
eps_end = 0.05
eps_decay = 0.995
eps = eps_start

# DS-DQN networks
main_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                  lr_enc=1e-4, lr_w=3e-3).to(device)
target_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                    lr_enc=1e-4, lr_w=3e-3).to(device)
target_net.encoder.load_state_dict(main_net.encoder.state_dict())
target_net.w.data.copy_(main_net.w.data)

replay = ReplayBuffer(10000)

# Logs
rewards_log, U_log, rho_log = [], [], []
td_log, corr_log, gen_log = [], [], []
w_norms, enc_norms = [], []

# λ schedule
LAMBDA_MAX = 0.02
DELTA_LAMBDA = LAMBDA_MAX / max(1, EPISODES)
lambda_reg = 0.0

for ep in range(EPISODES):
    idx = np.random.randint(len(TMS_train))
    TM = TMS_train[idx]
    tm_type = META_train.iloc[idx]["gen"]
    state = init_state(TM)

    ep_rewards, ep_Us, ep_rhos = [], [], []
    ep_td_losses, ep_corr_losses = [], []

    for step in range(K):
        # ε-greedy
        s_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_vals, _ = main_net.Q(s_tensor)
        if random.random() < eps:
            action = np.random.randint(N_ACTIONS)   # explore
        else:
            action = q_vals.argmax(1).item()        # exploit

        # env transition (now uses τ-filtering paths)
        s_next, r, U, rho = apply_action(state, action, TM, lam=0.5, tau=0.8)
        replay.push(state, action, r, s_next, 0.0)
        state = s_next

        ep_rewards.append(r); ep_Us.append(U); ep_rhos.append(rho)

        # learn
        if len(replay) >= BATCH_SIZE:
            batch = replay.sample(BATCH_SIZE)
            td, corr = main_net.update(batch, target_net, gamma=GAMMA, lambda_reg=lambda_reg)
            td_log.append(td); corr_log.append(corr)
            ep_td_losses.append(td); ep_corr_losses.append(corr)
            
            # Track parameter norms
            w_norms.append(main_net.w.detach().norm().item())
            enc_norms.append(sum(p.detach().norm().item() for p in main_net.encoder.parameters()))

    # episode logs
    rewards_log.append(float(np.mean(ep_rewards)))
    U_log.append(float(np.mean(ep_Us)))
    rho_log.append(float(np.mean(ep_rhos)))
    gen_log.append(tm_type)

    # ---- summary ----
    td_ep = np.mean(ep_td_losses) if ep_td_losses else float("nan")
    corr_ep = np.mean(ep_corr_losses) if ep_corr_losses else float("nan")
    print(
        f"[Ep {ep+1:04d}] "
        f"eps={eps:.3f} λ={lambda_reg:.4f} "
        f"R={np.mean(ep_rewards):+.4f} U={np.mean(ep_Us):.3f} ρ={np.mean(ep_rhos):.3f} "
        f"TD={td_ep:.5f} Corr={corr_ep:.5f} "
        f"replay={len(replay)}"
    )
    print(f"Episode {ep+1}, total routed flows: {K}, avg reward: {np.mean(ep_rewards):.4f}")
    
    # target net sync
    if (ep + 1) % TARGET_UPDATE == 0:
        target_net.encoder.load_state_dict(main_net.encoder.state_dict())
        target_net.w.data.copy_(main_net.w.data)

    # ramp λ
    lambda_reg = min(lambda_reg + DELTA_LAMBDA, LAMBDA_MAX)

    # decay ε
    eps = max(eps_end, eps * eps_decay)

# Save encoder φ(s)
phi_path = DATA_DIR / f"{TOPO_NAME}_phi_trained.pth"
torch.save(main_net.encoder.state_dict(), phi_path)
print(f"Saved encoder to {phi_path}")


In [ ]:
# Cell 7 — Inference (Algorithm 2, training-consistent with τ filtering + ε-decay per step + α decay)

def run_inference_alg2(
    TMS_test, META_test, phi_encoder,
    lam=0.55, episodes=1000, K=10,
    alpha0=1e-3, alpha_min=1e-5, alpha_decay=0.999,
    gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay=0.995
):
    """
    Inference (Alg 2):
    - Encoder φ(s) frozen; only w updated online with row-wise LFA.
    - ε-greedy with decay (decayed *per step*).
    - α decays per episode: large at start, smaller as adaptation continues.
    - τ filtering for path selection.
    """
    phi_encoder.eval()
    with torch.no_grad():
        phi_dim = phi_encoder(torch.zeros(1, STATE_DIM).to(device)).size(1)

    # Linear FA weights for all actions (start near 0 for stability)
    w = torch.zeros(N_ACTIONS, phi_dim, device=device)

    rewards_log, U_log, rho_log, gen_log = [], [], [], []

    # ε-decay init
    eps = eps_start
    from tqdm import trange

    for ep in trange(episodes, desc="Inference Episodes", ncols=100):

   # for ep in range(episodes):
        # 🔑 scheduled learning rate for this episode
        alpha = max(alpha_min, alpha0 * (alpha_decay ** ep))

        # sample a TM from test set
        idx = np.random.randint(len(TMS_test))
        TM = TMS_test[idx]
        tm_type = META_test.iloc[idx]["gen"]
        state = init_state(TM)

        ep_rewards, ep_Us, ep_rhos = [], [], []
        ep_deltas = []
        explore_ct = 0; exploit_ct = 0

        for step in range(K):
            # φ(s)
            s_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                phi_s = phi_encoder(s_tensor).squeeze(0)     # [d]

            # Q(s,·) = φ(s) @ w^T
            q_vals = torch.mv(w, phi_s)                      # [A]

            # ε-greedy with decaying ε
            if np.random.rand() < eps:
                action = np.random.randint(N_ACTIONS); explore_ct += 1
            else:
                action = int(torch.argmax(q_vals).item()); exploit_ct += 1

            # env step (τ filtering consistent with training)
            s_next, r, U, rho = apply_action(state, action, TM, lam=lam, tau=0.8)

            # TD target
            with torch.no_grad():
                phi_sp = phi_encoder(torch.FloatTensor(s_next).unsqueeze(0).to(device)).squeeze(0)
                q_next = torch.mv(w, phi_sp)
                y = torch.tensor(r, device=device) + gamma * torch.max(q_next)

            # δ = y - Q(s,a)
            q_sa = torch.dot(w[action], phi_s)
            delta = (y - q_sa).detach()
            ep_deltas.append(float(delta))

            # Row-wise update with decayed α
            with torch.no_grad():
                w[action].add_(alpha * delta * phi_s)

            # advance
            state = s_next
            ep_rewards.append(r); ep_Us.append(U); ep_rhos.append(rho)

            # ✅ decay ε per step
            eps = max(eps_end, eps * eps_decay)

        # episode logs
        Rm = float(np.mean(ep_rewards)); Um = float(np.mean(ep_Us)); rhom = float(np.mean(ep_rhos))
        rewards_log.append(Rm); U_log.append(Um); rho_log.append(rhom); gen_log.append(tm_type)

        # safe average for δ̄
        if not ep_deltas:
            d_mean = float('nan')
        else:
            d_mean = float(np.mean(ep_deltas))
    
        print(
            f"[Alg2 Ep {ep+1:03d}] λ={lam:.2f} eps={eps:.3f} α={alpha:.6f} "
            f"R={Rm:+.4f} U={Um:.3f} ρ={rhom:.3f} "
            f"δ̄={d_mean:+.5f} stepsE={explore_ct} stepsX={exploit_ct} tm={tm_type}"
        )

    return rewards_log, U_log, rho_log, gen_log


In [ ]:
class IdentityEncoder(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.state_dim = state_dim

    def forward(self, x):
        # Directly return raw state (no normalization)
        return x


In [ ]:
class RandomEncoder(nn.Module):
    def __init__(self, state_dim, phi_dim=None):
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim   # match transfer and scratch → same dim

        self.proj = nn.Linear(state_dim, phi_dim, bias=False)
        # freeze weights (no training)
        with torch.no_grad():
            self.proj.weight.copy_(torch.randn(phi_dim, state_dim) * 0.01)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        # Random projection without normalization
        return self.proj(x)


In [ ]:
# Cell 8 — Run inference: Transfer vs Scratch (with α schedule)

# 1) Load trained φ(s) and freeze
phi_trained = PhiEncoder(STATE_DIM, phi_dim=None).to(device)  # → 293
phi_trained.load_state_dict(torch.load(DATA_DIR / f"{TOPO_NAME}_phi_trained.pth",
                                       map_location=device))
phi_trained.eval()
for p in phi_trained.parameters():
    p.requires_grad = False

# 2) Scratch φ(s): identity mapping (raw state vector as φ(s))
phi_scratch = IdentityEncoder(STATE_DIM).to(device)
phi_scratch.eval()

# 3) Random φ(s): fixed random projection (now 293-dim too)
phi_random = RandomEncoder(STATE_DIM, phi_dim=None).to(device)
phi_random.eval()

# Common inference settings
LAM_TEST   = 0.7      # different operating point than training → good for transfer
EPISODES_I = 1000
K_I        = 30
GAMMA_I    = 0.99

# α schedules
ALPHA0_T   = 4.5e-5      # Transfer: small, stable
ALPHA_MIN_T = 1e-5
DECAY_T    = 0.999

ALPHA0_S   = 3e-4      # Scratch: still larger, but not exploding
ALPHA_MIN_S = 1e-5
DECAY_S    = 0.999

# TRANSFER run
rewards_T, U_T, rho_T, gen_T = run_inference_alg2(
    TMS_test, META_test, phi_trained,
    lam=LAM_TEST, episodes=EPISODES_I, K=K_I,
    alpha0=ALPHA0_T, alpha_min=ALPHA_MIN_T, alpha_decay=DECAY_T,
    gamma=GAMMA_I,
    eps_start=1.0, eps_end=0.05, eps_decay=0.995
)

# SCRATCH run
rewards_S, U_S, rho_S, gen_S = run_inference_alg2(
    TMS_test, META_test, phi_scratch,
    lam=LAM_TEST, episodes=EPISODES_I, K=K_I,
    alpha0=ALPHA0_S, alpha_min=ALPHA_MIN_S, alpha_decay=DECAY_S,
    gamma=GAMMA_I,
    eps_start=1.0, eps_end=0.05, eps_decay=0.995
)

# RANDOM run
rewards_R, U_R, rho_R, gen_R = run_inference_alg2(
    TMS_test, META_test, phi_random,
    lam=LAM_TEST, episodes=EPISODES_I, K=K_I,
    alpha0=ALPHA0_S, alpha_min=ALPHA_MIN_S, alpha_decay=DECAY_S,  # same as Scratch
    gamma=GAMMA_I,
    eps_start=1.0, eps_end=0.05, eps_decay=0.995
)


print(f"\n=== Summary (ε-greedy adaptation, λ={LAM_TEST:.2f}) ===")
print(f"TRANSFER:  R={np.mean(rewards_T):+.4f}, U={np.mean(U_T):.3f}, rho={np.mean(rho_T):.3f}")
print(f"SCRATCH :  R={np.mean(rewards_S):+.4f}, U={np.mean(U_S):.3f}, rho={np.mean(rho_S):.3f}")
print(f"RANDOM  :  R={np.mean(rewards_R):+.4f}, U={np.mean(U_R):.3f}, rho={np.mean(rho_R):.3f}")

print("φ_trained dim:", phi_trained(torch.zeros(1, STATE_DIM).to(device)).shape)
print("φ_scratch dim:", phi_scratch(torch.zeros(1, STATE_DIM).to(device)).shape)
print("φ_random dim :", phi_random(torch.zeros(1, STATE_DIM).to(device)).shape)


In [ ]:
# === Add-on Analysis: Threshold episode + EMA stats/plots (with consecutive check) ===
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Configurable settings ---
R_THRESH = 0.6     # reward threshold
N_CONSEC = 30      # must hold for these many consecutive episodes
BETA = 0.9         # EMA coefficient
WINDOW = 100
SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- Helper: find earliest consecutive streak crossing threshold ---
def find_first_above_threshold(rewards, thresh, consec=1):
    streak = 0
    for i, r in enumerate(rewards, start=1):
        if r >= thresh:
            streak += 1
            if streak >= consec:
                return i - consec + 1
        else:
            streak = 0
    return None

def format_ep(ep, total):
    return ep if ep is not None else f">{total}"

# --- Compute earliest episode for each baseline ---
ep_T = find_first_above_threshold(rewards_T, R_THRESH, N_CONSEC)
ep_S = find_first_above_threshold(rewards_S, R_THRESH, N_CONSEC)
ep_R = find_first_above_threshold(rewards_R, R_THRESH, N_CONSEC)
max_eps = EPISODES_I

print(f"\n=== Earliest Episode Reaching Reward ≥ {R_THRESH} for {N_CONSEC} consecutive ===")
print(f"Transfer: {format_ep(ep_T, max_eps)}")
print(f"Scratch : {format_ep(ep_S, max_eps)}")
print(f"Random  : {format_ep(ep_R, max_eps)}")

# --- EMA + windowed stats ---
def ema(series, beta=BETA):
    ema_vals = []
    s = series[0]
    for x in series:
        s = beta * s + (1 - beta) * x
        ema_vals.append(s)
    return np.array(ema_vals)

def last_ema_stats(rewards, U, rho, beta=BETA, window=WINDOW):
    r = ema(rewards, beta)[-window:]
    u = ema(U, beta)[-window:]
    rh = ema(rho, beta)[-window:]
    return np.mean(r), np.mean(u), np.mean(rh)

mR_T, mU_T, mRho_T = last_ema_stats(rewards_T, U_T, rho_T)
mR_S, mU_S, mRho_S = last_ema_stats(rewards_S, U_S, rho_S)
mR_R, mU_R, mRho_R = last_ema_stats(rewards_R, U_R, rho_R)

print(f"\n=== Last-{WINDOW} EMA Stats (β={BETA}) ===")
print(f"Transfer:  R={mR_T:+.4f}, U={mU_T:.3f}, ρ={mRho_T:.3f}")
print(f"Scratch :  R={mR_S:+.4f}, U={mU_S:.3f}, ρ={mRho_S:.3f}")
print(f"Random  :  R={mR_R:+.4f}, U={mU_R:.3f}, ρ={mRho_R:.3f}")

# --- Plot with EMA overlays ---
plt.figure(figsize=(10,5))
plt.plot(rewards_T, alpha=0.25, label="Transfer (raw)", color="tab:blue")
plt.plot(rewards_S, alpha=0.25, label="Scratch (raw)", color="tab:orange")
plt.plot(rewards_R, alpha=0.25, label="Random (raw)", color="tab:green")

plt.plot(ema(rewards_T), label=f"Transfer EMA (β={BETA})", color="tab:blue", lw=2)
plt.plot(ema(rewards_S), label=f"Scratch EMA (β={BETA})", color="tab:orange", lw=2)
plt.plot(ema(rewards_R), label=f"Random EMA (β={BETA})", color="tab:green", lw=2)

plt.axhline(R_THRESH, color="red", linestyle="--", lw=1,
            label=f"Threshold={R_THRESH} (≥{N_CONSEC} consecutive)")

plt.title(f"Reward with EMA smoothing (λ={LAM_TEST:.2f})")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fname = f"{SAVE_DIR}/reward_with_EMA.png"
plt.savefig(fname, dpi=150)
plt.show()
print(f"Saved: {fname}")


In [ ]:
# =======================================
# Cell 9 — Inference Results Visualization (Transfer vs Scratch vs Random)
# =======================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

ROLL_WIN = 100  # rolling window size for smoothing

def roll(x, w=ROLL_WIN):
    return pd.Series(x).rolling(window=w, min_periods=1).mean().to_numpy()

colors = {
    "Transfer": "tab:blue",
    "Scratch": "tab:orange",
    "Random":  "tab:green"
}

# --- 1️⃣ Reward ---
plt.figure(figsize=(10, 5))
plt.plot(rewards_T, alpha=0.25, color=colors["Transfer"], label="Transfer φ (raw)")
plt.plot(rewards_S, alpha=0.25, color=colors["Scratch"], label="Scratch φ (raw)")
plt.plot(rewards_R, alpha=0.25, color=colors["Random"],  label="Random φ (raw)")

plt.plot(roll(rewards_T), color=colors["Transfer"], lw=2, label=f"Transfer φ ({ROLL_WIN}-ep avg)")
plt.plot(roll(rewards_S), color=colors["Scratch"], lw=2, label=f"Scratch φ ({ROLL_WIN}-ep avg)")
plt.plot(roll(rewards_R), color=colors["Random"],  lw=2, label=f"Random φ ({ROLL_WIN}-ep avg)")

plt.axhline(0, lw=1, alpha=0.4, color="gray")
plt.title(f"Reward vs Episodes (λ={LAM_TEST:.2f})")
plt.xlabel("Episode"); plt.ylabel("Reward")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
fname = f"{SAVE_DIR}/reward_overall.png"
plt.savefig(fname, dpi=150); plt.show()
print(f"Saved: {fname}")

# --- 2️⃣ Max Utilization U ---
plt.figure(figsize=(10, 5))
plt.plot(U_T, alpha=0.25, color=colors["Transfer"], label="Transfer (raw)")
plt.plot(U_S, alpha=0.25, color=colors["Scratch"], label="Scratch (raw)")
plt.plot(U_R, alpha=0.25, color=colors["Random"],  label="Random (raw)")

plt.plot(roll(U_T), color=colors["Transfer"], lw=2, label=f"Transfer ({ROLL_WIN}-ep avg)")
plt.plot(roll(U_S), color=colors["Scratch"], lw=2, label=f"Scratch ({ROLL_WIN}-ep avg)")
plt.plot(roll(U_R), color=colors["Random"],  lw=2, label=f"Random ({ROLL_WIN}-ep avg)")

plt.ylim(bottom=0)
plt.title("Max Utilization U vs Episodes")
plt.xlabel("Episode"); plt.ylabel("U")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
fname = f"{SAVE_DIR}/maxU_overall.png"
plt.savefig(fname, dpi=150); plt.show()
print(f"Saved: {fname}")

# --- 3️⃣ Average Utilization ρ̄ ---
plt.figure(figsize=(10, 5))
plt.plot(rho_T, alpha=0.25, color=colors["Transfer"], label="Transfer (raw)")
plt.plot(rho_S, alpha=0.25, color=colors["Scratch"], label="Scratch (raw)")
plt.plot(rho_R, alpha=0.25, color=colors["Random"],  label="Random (raw)")

plt.plot(roll(rho_T), color=colors["Transfer"], lw=2, label=f"Transfer ({ROLL_WIN}-ep avg)")
plt.plot(roll(rho_S), color=colors["Scratch"], lw=2, label=f"Scratch ({ROLL_WIN}-ep avg)")
plt.plot(roll(rho_R), color=colors["Random"],  lw=2, label=f"Random ({ROLL_WIN}-ep avg)")

plt.ylim(bottom=0)
plt.title("Average Utilization ρ̄ vs Episodes")
plt.xlabel("Episode"); plt.ylabel("ρ̄")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
fname = f"{SAVE_DIR}/avgRho_overall.png"
plt.savefig(fname, dpi=150); plt.show()
print(f"Saved: {fname}")


In [ ]:
# Build combined DataFrame for per-TM analysis (Transfer, Scratch, Random)
import pandas as pd
import numpy as np

df_T = pd.DataFrame({
    "episode": np.arange(len(rewards_T)),
    "gen": gen_T,
    "reward": rewards_T,
    "U": U_T,
    "rho": rho_T,
    "mode": "Transfer"
})

df_S = pd.DataFrame({
    "episode": np.arange(len(rewards_S)),
    "gen": gen_S,
    "reward": rewards_S,
    "U": U_S,
    "rho": rho_S,
    "mode": "Scratch"
})

df_R = pd.DataFrame({
    "episode": np.arange(len(rewards_R)),
    "gen": gen_R,
    "reward": rewards_R,
    "U": U_R,
    "rho": rho_R,
    "mode": "Random"
})

# Combine all three baselines
df_all = pd.concat([df_T, df_S, df_R], ignore_index=True)


In [ ]:
import os

# create a directory to save
SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

tm_types = sorted(df_all["gen"].unique())
metrics = [("reward","Reward"), ("U","Max U"), ("rho","Avg ρ̄")]

# dynamically grab all modes present (Transfer, Scratch, Random, etc.)
modes = sorted(df_all["mode"].unique())

for met_key, met_label in metrics:
    fig, axes = plt.subplots(1, len(tm_types), figsize=(5*len(tm_types), 3), sharey=False)
    if len(tm_types) == 1:
        axes = [axes]
    for ax, g in zip(axes, tm_types):
        sub = df_all[df_all["gen"] == g].copy()
        for mode in modes:
            ss = sub[sub["mode"] == mode].sort_values("episode")
            ax.plot(ss["episode"], ss[met_key], alpha=0.25, label=f"{mode} (raw)")
            ax.plot(ss["episode"], ss[met_key].rolling(100, min_periods=1).mean(),
                    label=f"{mode} (100-ep avg)")
        ax.set_title(f"{met_label} — {g} (λ=0.7)")
        ax.set_xlabel("Episode"); ax.set_ylabel(met_label)
        ax.grid(True, alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(modes)*2)  # adapt legend columns
    fig.tight_layout(rect=[0,0,1,0.92])

    # save automatically
    fname = f"{SAVE_DIR}/{met_key}_by_tmtype.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Saved: {fname}")


In [ ]:
# =======================================
# Summary statistics and % improvements
# =======================================

import pandas as pd
import numpy as np

# --- Stats helper functions ---
def last_stats(x, k=None):
    """Return mean ± std for last k episodes, or full if k=None."""
    x = np.asarray(x)
    if k is None or k > len(x):
        x = x[:]  # full
    else:
        x = x[-k:]
    return float(np.mean(x)), float(np.std(x))

def fmt_stats(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

# --- Collect stats for all modes dynamically ---
n_eps = len(rewards_T)  # assumes all runs have the same length
windows = {"last100": 100, "last200": 200, f"all{n_eps}": None}

rows = []
all_results = {
    "Transfer": (rewards_T, U_T, rho_T),
    "Scratch":  (rewards_S, U_S, rho_S),
    "Random":   (rewards_R, U_R, rho_R)
}

# --- Compute mean ± std for each mode ---
for mode, (R, U, Rho) in all_results.items():
    row = {"mode": mode}
    for label, k in windows.items():
        mR, sR = last_stats(R, k)
        mU, sU = last_stats(U, k)
        mRho, sRho = last_stats(Rho, k)
        row[f"Reward_{label}"] = fmt_stats(mR, sR)
        row[f"U_{label}"]      = fmt_stats(mU, sU)
        row[f"rho_{label}"]    = fmt_stats(mRho, sRho)
    rows.append(row)

# --- Compute % improvements (Transfer over Scratch and Random) ---
for base in ["Scratch", "Random"]:
    row_impr = {"mode": f"% Improvement (T over {base[0]})"}
    for label, k in windows.items():
        meanT, _ = last_stats(rewards_T, k)
        meanB_R, _ = last_stats(all_results[base][0], k)
        meanT_U, _ = last_stats(U_T, k)
        meanB_U, _ = last_stats(all_results[base][1], k)
        meanT_Rho, _ = last_stats(rho_T, k)
        meanB_Rho, _ = last_stats(all_results[base][2], k)

        # Reward (↑ higher better)
        impr_reward = 100 * (meanT - meanB_R) / meanB_R if meanB_R != 0 else float("nan")
        # U, ρ̄ (↓ lower better)
        impr_U = 100 * (meanB_U - meanT_U) / meanB_U if meanB_U != 0 else float("nan")
        impr_rho = 100 * (meanB_Rho - meanT_Rho) / meanB_Rho if meanB_Rho != 0 else float("nan")

        row_impr[f"Reward_{label}"] = f"{impr_reward:.2f}%"
        row_impr[f"U_{label}"]      = f"{impr_U:.2f}%"
        row_impr[f"rho_{label}"]    = f"{impr_rho:.2f}%"

    rows.append(row_impr)

# --- Display summary table ---
summary = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("\n=== Inference Summary ===")
print(summary)

# --- Optional: save to CSV for paper appendix ---
summary.to_csv("./plots/inference_summary.csv", index=False)
print("\nSaved: ./plots/inference_summary.csv")
